# Historical Analysis — F25 / F26 / F27 Toolkits

**Persona:** Marketing Operations Analyst

**Questions:**
- How does the pre-buy → final-outcome funnel look?
- Where is spend concentrated by brand and season?
- How many items get cancelled vs swapped to POD vs fulfilled from inventory?
- Who are our highest-volume / most reliable vendors?

**Data source:** Supabase Postgres (project `YOUR-PROJECT-REF`), reading from views created in `supabase/migrations/20260531000004_historical_analytics_views.sql`:
- `v_item_outcomes` — one row per toolkit_item, source of truth
- `v_prebuy_funnel` — workflow funnel counts/spend
- `v_spend_by_brand_season`
- `v_spend_by_vendor_season`
- `v_cancellation_by_dimension`

All reports below mirror the live dashboard at `site/historical-analytics.html` — same views, same numbers. Use this notebook for one-off analytical cuts the dashboard doesn't expose.

## Setup
Reads `SUPABASE_URL` and `SUPABASE_ANON` from a `.env` file at the repo root. If you don't have one, create it from `.env.example`. The anon key is read-only via RLS.

```
pip install supabase python-dotenv pandas matplotlib seaborn
```

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from supabase import create_client

# Load env vars from repo root (notebook lives in notebooks/)
load_dotenv(Path('..').resolve() / '.env')
SUPABASE_URL  = os.environ['SUPABASE_URL']
SUPABASE_ANON = os.environ['SUPABASE_ANON']

sb = create_client(SUPABASE_URL, SUPABASE_ANON)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
PALETTE = {
    'approved':            '#1A7A4A',
    'inventory_fulfilled': '#0D7377',
    'cancel_cancelled':    '#C0392B',
    'cancel_pod':          '#D49C2A',
    'removed_prebuy':      '#A0522D',
    'pod_prebuy':          '#C8973A',
    'abc_merch_prebuy':    '#8C28A8',
    'in_flight_requoting': '#7a8699',
    'no_outcome':          '#cdd2d8',
    'part_of_kit':         '#3B6FD4',
    'unknown':             '#aaa',
}

def fetch(view, columns='*'):
    """Pull a Supabase view into a DataFrame (handles pagination automatically up to 1000 rows)."""
    res = sb.from_(view).select(columns).range(0, 4999).execute()
    return pd.DataFrame(res.data)

outcomes      = fetch('v_item_outcomes')
funnel        = fetch('v_prebuy_funnel')
brand_season  = fetch('v_spend_by_brand_season')
vendor_season = fetch('v_spend_by_vendor_season')
cancel_dim    = fetch('v_cancellation_by_dimension')

# Coerce numerics (Supabase JSON sometimes returns numeric as str)
for df in (outcomes, funnel, brand_season, vendor_season, cancel_dim):
    for c in df.columns:
        if df[c].dtype == 'object':
            try: df[c] = pd.to_numeric(df[c])
            except (ValueError, TypeError): pass

print(f'item outcomes: {len(outcomes):>5}   funnel rows: {len(funnel):>5}   brand-season: {len(brand_season):>5}   vendor-season: {len(vendor_season):>5}   cancel-dim: {len(cancel_dim):>5}')
outcomes.head(3)

## Report 1 — Pre-Buy Funnel
Final outcome breakdown by fiscal year. Same shape as Report 01 on the dashboard.

In [ ]:
pivot = (outcomes
         .groupby(['fiscal_year','final_outcome']).size()
         .unstack(fill_value=0))

# Reorder outcomes most-favourable → least
outcome_order = ['approved','inventory_fulfilled','part_of_kit','in_flight_requoting','no_outcome',
                 'cancel_cancelled','cancel_pod','removed_prebuy','pod_prebuy','abc_merch_prebuy','unknown']
cols = [c for c in outcome_order if c in pivot.columns]
pivot = pivot[cols]

ax = pivot.plot(kind='bar', stacked=True, figsize=(10,4.5),
                color=[PALETTE.get(c,'#999') for c in pivot.columns],
                edgecolor='white', linewidth=0.5)
ax.set_title('Pre-Buy Funnel — Final Outcome by Fiscal Year', fontsize=14, pad=12)
ax.set_xlabel('')
ax.set_ylabel('Item count')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9, title='Final outcome')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Rate table: approval rate = approved / (approved+inventory+cancelled)
considered = ['approved','inventory_fulfilled','cancel_cancelled','cancel_pod']
rates = (outcomes
         .assign(in_consideration = outcomes['final_outcome'].isin(considered),
                 is_approved      = outcomes['final_outcome'].eq('approved'),
                 is_cancelled     = outcomes['final_outcome'].isin(['cancel_cancelled','cancel_pod']))
         .groupby('fiscal_year')
         .agg(items=('toolkit_item_id','count'),
              considered=('in_consideration','sum'),
              approved=('is_approved','sum'),
              cancelled=('is_cancelled','sum')))
rates['approval_rate']     = (rates['approved']  / rates['considered']).round(3)
rates['cancellation_rate'] = (rates['cancelled'] / rates['considered']).round(3)
rates

## Report 2 — Cancellation Deep-Dive
Breakdown by brand, item type, vendor; true cancels vs POD swaps.

In [ ]:
# Pull just the 'brand' slice from the cancellation-by-dimension view
cd_brand = (cancel_dim[cancel_dim['dimension_type'].eq('brand')]
            .groupby('dimension_value')
            .agg(items=('total_items','sum'),
                 true_cancels=('true_cancels','sum'),
                 pod_swaps=('pod_swaps','sum'),
                 cancel_spend=('cancelled_spend','sum'))
            .assign(cancel_rate=lambda d: ((d['true_cancels']+d['pod_swaps'])/d['items']).round(3))
            .sort_values('cancel_rate', ascending=False)
            .head(15))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: stacked counts
cd_brand[['true_cancels','pod_swaps']].plot(kind='barh', stacked=True, ax=axes[0],
    color=[PALETTE['cancel_cancelled'], PALETTE['cancel_pod']],
    edgecolor='white', linewidth=0.5)
axes[0].set_title('Top brands by cancellation rate — counts', fontsize=12)
axes[0].set_xlabel('Cancelled items'); axes[0].set_ylabel('')
axes[0].invert_yaxis()

# Right: dollar exposure
(cd_brand['cancel_spend']/1000).plot(kind='barh', ax=axes[1], color=PALETTE['cancel_cancelled'], edgecolor='white')
axes[1].set_title('Dollar exposure on cancelled items ($K)', fontsize=12)
axes[1].set_xlabel('$K'); axes[1].set_ylabel('')
axes[1].invert_yaxis()

plt.tight_layout(); plt.show()
cd_brand

## Report 3 — Total Spend Over Time
Budget (revised snapshot) vs actual (final.production_spend), stacked by item category. F25 has no budget data — flagged as data gap.

In [ ]:
spend = (outcomes
         .groupby(['fiscal_year','item_category'])
         .agg(budget=('budget_spend','sum'),
              actual=('final_production_spend','sum'))
         .fillna(0)
         .reset_index())

# Two-bar (budget vs actual) per year, stacked by category
import numpy as np
years = sorted(spend['fiscal_year'].unique())
cats  = ['paper','display','premium']
x = np.arange(len(years))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 4.5))
bottoms_b = np.zeros(len(years)); bottoms_a = np.zeros(len(years))
for c, color in zip(cats, ['#3B6FD4','#0D7377','#8C28A8']):
    bvals = [spend.query('fiscal_year==@y and item_category==@c')['budget'].sum()/1000 for y in years]
    avals = [spend.query('fiscal_year==@y and item_category==@c')['actual'].sum()/1000 for y in years]
    ax.bar(x-w/2, bvals, w, bottom=bottoms_b, label=f'budget · {c}',  color=color, alpha=0.45, edgecolor='white')
    ax.bar(x+w/2, avals, w, bottom=bottoms_a, label=f'actual · {c}', color=color, edgecolor='white')
    bottoms_b += bvals; bottoms_a += avals

ax.set_xticks(x); ax.set_xticklabels(years)
ax.set_ylabel('$K'); ax.set_title('Budget vs Actual Spend by Fiscal Year (stacked by category)')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9, ncol=1)
plt.tight_layout(); plt.show()

# Variance table
totals = spend.groupby('fiscal_year').agg(budget=('budget','sum'), actual=('actual','sum'))
totals['variance']     = totals['actual'] - totals['budget']
totals['variance_pct'] = (totals['variance']/totals['budget'].replace(0, pd.NA)).round(3)
totals.style.format({'budget':'${:,.0f}','actual':'${:,.0f}','variance':'${:,.0f}','variance_pct':'{:+.1%}'})

## Report 4 — Spend by Brand (F25 vs F26)
Top brands side-by-side.

In [ ]:
bs = brand_season[brand_season['fiscal_year'].isin(['F25','F26'])].copy()
bs['actual_spend'] = bs['actual_spend'].astype(float)

pivot = bs.pivot_table(index='brand_name', columns='fiscal_year',
                       values='actual_spend', aggfunc='sum').fillna(0)
pivot['_total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('_total', ascending=True).drop(columns='_total').tail(15)

ax = (pivot/1000).plot(kind='barh', figsize=(10, 6),
                       color=['#7a8699', PALETTE['approved']],
                       edgecolor='white')
ax.set_title('Top 15 brands — actual spend F25 vs F26 ($K)')
ax.set_xlabel('$K'); ax.set_ylabel('')
ax.legend(title='Fiscal year')
plt.tight_layout(); plt.show()

# Brand-level cancel-rate table
considered_ids = outcomes['final_outcome'].isin(['approved','inventory_fulfilled','cancel_cancelled','cancel_pod'])
cancel_ids     = outcomes['final_outcome'].isin(['cancel_cancelled','cancel_pod'])
brand_summary = (outcomes
                 .groupby(['brand_name','fiscal_year'])
                 .agg(items=('toolkit_item_id','count'),
                      considered=('final_outcome', lambda s: s.isin(['approved','inventory_fulfilled','cancel_cancelled','cancel_pod']).sum()),
                      cancelled=('final_outcome', lambda s: s.isin(['cancel_cancelled','cancel_pod']).sum()),
                      actual=('final_production_spend','sum'),
                      budget=('budget_spend','sum'))
                 .assign(cancel_rate=lambda d: (d['cancelled']/d['considered']).round(3))
                 .query('fiscal_year in ["F25","F26"]')
                 .reset_index())
brand_summary.head(20)

## Report 5 — Vendor Patterns
Volume, spend, POD-swap rate.

In [ ]:
vs = (vendor_season
      .groupby('vendor_name')
      .agg(items=('item_count','sum'),
           actual=('actual_spend','sum'),
           pod_swaps=('items_pod_swapped','sum'),
           approved=('items_approved','sum'))
      .assign(pod_rate=lambda d: (d['pod_swaps']/(d['approved']+d['pod_swaps']).replace(0, pd.NA)).round(3))
      .sort_values('actual', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
(vs['items']).plot(kind='bar', ax=axes[0], color=PALETTE['approved'], edgecolor='white')
axes[0].set_title('Items per vendor'); axes[0].set_ylabel('Items')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

(vs['actual']/1000).plot(kind='bar', ax=axes[1], color='#0D7377', edgecolor='white')
axes[1].set_title('Actual spend per vendor ($K)'); axes[1].set_ylabel('$K')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
plt.tight_layout(); plt.show()
vs

## Report 6 — Item Type & Spec Mix
Counts by item type plus Standard/Custom and New/Rerun splits.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Top 10 item types
top_types = outcomes['item_type'].value_counts().head(10)
top_types.plot(kind='barh', ax=axes[0], color=PALETTE['approved'], edgecolor='white')
axes[0].set_title('Top 10 item types'); axes[0].invert_yaxis()

# Standard vs Custom
sc = outcomes['standard_or_custom'].fillna('Unknown').value_counts()
sc.plot(kind='pie', ax=axes[1], autopct='%1.0f%%', colors=['#1A7A4A','#D49C2A','#cdd2d8'], wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Standard vs Custom'); axes[1].set_ylabel('')

# New vs Rerun
nr = outcomes['new_or_rerun'].fillna('Unknown').value_counts()
nr.plot(kind='pie', ax=axes[2], autopct='%1.0f%%', colors=['#3B6FD4','#1A7A4A','#cdd2d8'], wedgeprops={'edgecolor':'white','linewidth':2})
axes[2].set_title('New vs Rerun'); axes[2].set_ylabel('')

plt.tight_layout(); plt.show()

## Custom Query Scratchpad
Examples below — adapt as needed. The `outcomes` DataFrame is the source of truth (mirrors `v_item_outcomes`).

Common cuts the dashboard doesn't expose:

In [ ]:
# Cancellation rate by lead-time bucket
(outcomes
  .assign(considered=outcomes['final_outcome'].isin(['approved','inventory_fulfilled','cancel_cancelled','cancel_pod']),
          cancelled =outcomes['final_outcome'].isin(['cancel_cancelled','cancel_pod']))
  .groupby(outcomes['lead_time'].fillna('Unknown'))
  .agg(items=('toolkit_item_id','count'),
       considered=('considered','sum'),
       cancelled=('cancelled','sum'))
  .assign(cancel_rate=lambda d: (d['cancelled']/d['considered']).round(3))
  .sort_values('items', ascending=False))

In [ ]:
# Inventory partial fulfillment by brand — how much demand was met without producing?
(outcomes[outcomes['inventory_spend_partial']]
  .groupby(['brand_name','fiscal_year'])
  .agg(items=('toolkit_item_id','count'),
       inventory_spend=('final_inventory_spend','sum'),
       production_spend=('final_production_spend','sum'))
  .assign(inventory_share=lambda d: (d['inventory_spend']/(d['inventory_spend']+d['production_spend'])).round(3))
  .sort_values('inventory_spend', ascending=False)
  .head(15))

In [ ]:
# Brand × buy-wave heatmap of item counts — where is each brand most active?
hm = (outcomes
      .pivot_table(index='brand_name', columns='buy_wave', values='toolkit_item_id', aggfunc='count', fill_value=0))
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(hm, annot=True, fmt='d', cmap='YlGnBu', ax=ax, cbar_kws={'label':'Items'})
ax.set_title('Items per brand × buy-wave')
plt.tight_layout(); plt.show()

## Data Gaps & Caveats
- **F25 has no `budget_spend`** — only `final_production_spend`. The buy was completed before revised-snapshot budgets were tracked. Variance analysis is F26/F27 only.
- **No event-level timestamps** — we can see *what* happened, not *when*. Cancellation-by-month requires an event_log table that doesn't exist yet.
- **Demo dataset** — brands are mixed real + anonymized. Treat absolute numbers as illustrative; relative patterns are real.
- **Inventory fulfillment** is signalled by `inventory_spend > 0`, not a status string. Items with `final_outcome='approved'` may *also* have partial inventory fulfillment — they're not mutually exclusive.